In [1]:
#import the necessary packages for all analysis
import pandas as pd
import numpy as np 

In [2]:
#Import the data from Kaggle 
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/test.csv')
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/train.csv')

# Decision Trees

In [3]:
#now import the sklearn packages for splitting the sample and for running the Logit analysis 
#for splitting test/train and for k-folds cross validation 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV 

#to make transforming the data easier ColumnTransformer and Pipeline keep preprocessed data inside the folds 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#for encoding the data 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#to actually run the model 
#from sklearn.linear_model import LogisticRegression 
#from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

#to test how well the model performs 
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score 

#to see which features are most important 
from sklearn.inspection import permutation_importance

In [4]:
#first create additional features
def add_titanic_features(df: pd.DataFrame) -> pd.DataFrame: 
    df = df.copy()

    #add log(fare) 
    df["log_Fare"] = np.log1p(df["Fare"])

    #family features (total size and if they're alone) 
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    #title from name 
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand = False).str.strip()
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss", "Ms":"Miss","Mme":"Mrs"
    })
    df["Title"] = df["Title"].where(df["Title"].isin(["Mr","Mrs","Miss","Master"]), "Other")

    #Cabin Features 
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")

    return df

In [5]:
#now add the additional covariates to the training data 
train_fe = add_titanic_features(train_df)

#define the feature list, then the X and y 
Feature_List = [
    "Pclass", "Sex", "Age",
    "log_Fare", "Embarked",
    "FamilySize", "IsAlone",
    "Title",
    "CabinKnown", "Deck",
]

X = train_fe[Feature_List]
y = train_fe["Survived"]

In [6]:
#now build the preprocessing 
numeric_features = ["Age","log_Fare","FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked",  "Title", "Deck"]
binary_features = ["IsAlone","CabinKnown"]
#numeric preprocessing
numeric_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy  = "median", add_indicator = True)),
                                         ]) # don't need scaling for RF

#categorical preprocessing 
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "Missing")),
                                             ("onehot", OneHotEncoder(handle_unknown = "ignore"))]) # don't need to drop for RF

#binary preprocessing 
binary_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent", add_indicator = True))])
#preprocess 
preprocess = ColumnTransformer(transformers = [("num", numeric_transformer, numeric_features),
                                              ("cat",categorical_transformer, categorical_features),
                                              ("bin", binary_transformer, binary_features)])


## Decision Tree with Simple 80-20 split

In [12]:
#split the sample
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 1234, stratify = y)

#define the decision tree model with no hyperparameter tuning
decision_tree_model = DecisionTreeClassifier(random_state = 1234)

decision_tree = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", decision_tree_model)])

#fit the model 
decision_tree.fit(X_train, y_train)

#Evaluate the model
probability_tree = decision_tree.predict_proba(X_val)[:,1]
prediction_tree = (probability_tree > 0.5).astype(int)

#RF Evaluation 
tree_eval = {
    "log_loss": log_loss(y_val, probability_tree),
    "roc_auc": roc_auc_score(y_val, probability_tree),
    "accuracy": accuracy_score(y_val, prediction_tree)
}
print("Decision Tree:" ,tree_eval)
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

decision_tree_model_restricted = DecisionTreeClassifier(random_state = 1234, 
                                            max_depth=3,
                                            min_samples_leaf=10,
                                            min_samples_split=20)
decision_tree_restricted = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", decision_tree_model_restricted)])

#fit the model 
decision_tree_restricted.fit(X_train, y_train)

#Evaluate the model
probability_tree_restricted = decision_tree_restricted.predict_proba(X_val)[:,1]
prediction_tree_restricted = (probability_tree_restricted > 0.5).astype(int)

#RF Evaluation 
tree_eval_restricted = {
    "log_loss": log_loss(y_val, probability_tree_restricted),
    "roc_auc": roc_auc_score(y_val, probability_tree_restricted),
    "accuracy": accuracy_score(y_val, prediction_tree_restricted)
}
print("Decision Tree (restricted):" ,tree_eval_restricted)


Decision Tree: {'log_loss': 8.265248680135993, 'roc_auc': 0.7457180500658762, 'accuracy': 0.770949720670391}
Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}
Decision Tree (restricted): {'log_loss': 0.8545450325555427, 'roc_auc': 0.8210144927536231, 'accuracy': 0.8100558659217877}


In [16]:
#Next I create a tree and hypertune the parameters. 
tree = DecisionTreeClassifier(random_state = 1234)

decision_tree_tuned = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", decision_tree_model)])
#next set the parameter grid i want to search over and i'm going to search over paramters, evaluation criterion, and pruning 
param_grid = {"model__max_depth": [None, 2, 3, 5, 8, 12],
    "model__min_samples_leaf": [1, 2, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__criterion": ["gini", "entropy"], #evaluation criterion
    "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005] #pruning
}

inner_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 1234) 

tree_search = GridSearchCV(decision_tree_tuned, 
                          param_grid = param_grid, 
                          scoring = "neg_log_loss", 
                          cv = inner_cv, 
                          n_jobs = -1)

tree_search.fit(X_train,y_train)
best_tree = tree_search.best_estimator_

print("Best params:", tree_search.best_params_)
print("Best CV score (train folds):", tree_search.best_score_)

# Evaluate tuned tree on holdout
probability_tuned = best_tree.predict_proba(X_val)[:, 1]
prediction_tuned  = (probability_tuned >= 0.5).astype(int)

tree_eval_tuned = {
    "log_loss": log_loss(y_val, probability_tuned),
    "roc_auc": roc_auc_score(y_val, probability_tuned),
    "accuracy": accuracy_score(y_val, prediction_tuned)
}
print("Decision Tree (tuned) holdout:", tree_eval_tuned)


Best params: {'model__ccp_alpha': 0.0, 'model__criterion': 'entropy', 'model__max_depth': 2, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Best CV score (train folds): -0.46686106111760195
Decision Tree (tuned) holdout: {'log_loss': 0.46218955220679664, 'roc_auc': 0.8381422924901187, 'accuracy': 0.7821229050279329}


## Decision Tree with K-Fold Cross Validation and Hyperparameter Tuning

In [17]:
#now do k-fold and hyperparameter tuning
outer_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 1234) 
inner_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 1234) 

#Parameter Grid same as before
param_grid = {"model__max_depth": [None, 2, 3, 5, 8, 12],
    "model__min_samples_leaf": [1, 2, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__criterion": ["gini", "entropy"], #evaluation criterion
    "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005] #pruning
}

#define the Tree pipeline
tree_tune_kfold = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", decision_tree_model)])

tree_tune_search_kfold = GridSearchCV(tree_tune_kfold, 
                          param_grid = param_grid, 
                          scoring = "neg_log_loss", 
                          cv = inner_cv, 
                          n_jobs = -1)
#now run the cross validation 
scoring = {
    'logloss': 'neg_log_loss',
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy'}

cv_results_tree_tune = cross_validate(
    tree_tune_search_kfold, X, y, 
    cv = outer_cv, 
    scoring = scoring, #same scoring as above with the logit
    return_train_score = False 
)
#now calculate and print out the model quality 
print("Logit K-Fold Cross Validation Evaluation", "\n", "5-fold CV Log Loss: 0.4357 ± 0.0067","\n","5-fold CV ROC-AUC:  0.8695 ± 0.0182",
    "\n","5-fold CV Accuracy: 0.8261 ± 0.0276")
print("Nested tree Hypertune")
print("Log loss:", (-cv_results_tree_tune["test_logloss"]).mean(), "±", (-cv_results_tree_tune["test_logloss"]).std())
print("ROC-AUC: ", (cv_results_tree_tune["test_roc_auc"]).mean(), "±", (cv_results_tree_tune["test_roc_auc"]).std())
print("Accuracy:", (cv_results_tree_tune["test_accuracy"]).mean(), "±", (cv_results_tree_tune["test_accuracy"]).std())

Logit K-Fold Cross Validation Evaluation 
 5-fold CV Log Loss: 0.4357 ± 0.0067 
 5-fold CV ROC-AUC:  0.8695 ± 0.0182 
 5-fold CV Accuracy: 0.8261 ± 0.0276
Nested tree Hypertune
Log loss: 0.4693447953332458 ± 0.08501807103419913
ROC-AUC:  0.8550250475140195 ± 0.024762094586837004
Accuracy: 0.8058439520431863 ± 0.011422394224426107
